# Sneha Singh
## DATA 266 — Homework 6
Self-supervised and contrastive representation learning on STL-10

Same backbone for every part: ResNet-18, no pretrained weights.


In [ ]:
# libraries for this homework
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import models, transforms
from torchvision.datasets import STL10

# last 4 of my SJSUid - 0670
SID4 = 670
SEED = SID4
SLICE = SID4 % 1000
HP_ID = SID4 % 6
CLS_A = SID4 % 10
CLS_B = (CLS_A + 1 + ((SID4 // 10) % 9)) % 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

ROOT = Path("data")
CKPT = Path("checkpoints")
FIG = Path("figures")
ROOT.mkdir(exist_ok=True)
CKPT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

print("SID4:", SID4)
print("SEED:", SEED)
print("SLICE:", SLICE)
print("HP_ID:", HP_ID)
print("CLS_A:", CLS_A)
print("CLS_B:", CLS_B)
print("device:", DEVICE)
print("HP_ID is reported only. no extra HP model this hw")
print("three models = supervised, rotation SSL, SimCLR")


This assignment said follow the same step 0 as assignment 1. I did not copy the HW1 diabetes notebook. I only reused the standing parameters: SID4=670, SEED=670, SLICE=670, HP_ID=4, CLS_A=0, CLS_B=5, and I seeded python / numpy / torch with 670.

HW1 used HP_ID for a second network (15 vs 30 epochs). This assignment said skip that, because the three models are already the supervised net, the rotation encoder, and SimCLR.

Run this on a Colab GPU. Part A is only 500 images. Part B is all 100,000 unlabeled images for 15 epochs. Part C is 20,000 unlabeled images, two views, for 15 epochs. That is too long for my Mac.


## 0. Data

STL-10. Train split is 5,000 labeled images. I take 10% of that, so 500 images, and I reuse that same index list in Parts A, B, and C. Test split is 8,000 images. Unlabeled split is 100,000 images. Part C uses the first 20,000 of a seed-670 shuffle of the unlabeled split.


In [ ]:
STL_MEAN = (0.4467, 0.4398, 0.4066)
STL_STD = (0.2603, 0.2566, 0.2713)

eval_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(STL_MEAN, STL_STD),
])

train_labeled = STL10(ROOT, split="train", download=True, transform=eval_tf)
test_set = STL10(ROOT, split="test", download=True, transform=eval_tf)
unlabeled_raw = STL10(ROOT, split="unlabeled", download=True)

g = torch.Generator().manual_seed(SEED)
labeled_idx = torch.randperm(len(train_labeled), generator=g)[:500].tolist()
unlab_perm = torch.randperm(len(unlabeled_raw), generator=g)[:20000].tolist()

torch.save(labeled_idx, CKPT / "labeled_10pct_idx.pt")
torch.save(unlab_perm, CKPT / "simclr_20k_idx.pt")

print("labeled train:", len(train_labeled))
print("10% subset:", len(labeled_idx))
print("test:", len(test_set))
print("unlabeled:", len(unlabeled_raw))
print("simclr subset:", len(unlab_perm))
print("classes:", train_labeled.classes)


In [ ]:
def make_encoder():
    net = models.resnet18(weights=None)
    net.fc = nn.Identity()
    return net

def count_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad)

@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    for x, y in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)
        pred = model(x).argmax(1)
        correct += (pred == y).sum().item()
        total += y.numel()
    return correct / total

def run_epochs(model, loader, opt, epochs, eval_loader=None, name="", contrastive=False):
    rows = []
    for epoch in range(1, epochs + 1):
        model.train()
        if contrastive and hasattr(model, "encoder"):
            # BN stats stay fixed only during the linear probe, not here
            pass
        total = 0.0
        n = 0
        for batch in loader:
            if contrastive:
                v1, v2 = batch
                v1 = v1.to(DEVICE)
                v2 = v2.to(DEVICE)
                loss = model(v1, v2)
                bs = v1.size(0)
            else:
                x, y = batch
                x = x.to(DEVICE)
                y = y.to(DEVICE)
                loss = F.cross_entropy(model(x), y)
                bs = y.size(0)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item() * bs
            n += bs
        acc = accuracy(model, eval_loader) if eval_loader is not None else None
        row = {"epoch": epoch, "loss": total / n, "acc": acc}
        rows.append(row)
        if acc is None:
            print(f"{name} epoch {epoch:02d}  loss {row['loss']:.4f}")
        else:
            print(f"{name} epoch {epoch:02d}  loss {row['loss']:.4f}  acc {acc:.4f}")
    return rows


## Part A — Supervised, 10% labels

I train ResNet-18 end to end on the 500-image subset for 12 epochs. The last layer is a linear classifier with 10 classes. Test accuracy is on the full 8,000-image test split.


In [ ]:
class SupervisedNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = make_encoder()
        self.head = nn.Linear(512, 10)

    def forward(self, x):
        return self.head(self.encoder(x))

    def embed(self, x):
        return self.encoder(x)

labeled_subset = Subset(train_labeled, labeled_idx)
train_loader_a = DataLoader(labeled_subset, batch_size=64, shuffle=True, num_workers=2)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False, num_workers=2)

torch.manual_seed(SEED)
model_a = SupervisedNet().to(DEVICE)
opt_a = torch.optim.Adam(model_a.parameters(), lr=1e-3)
print("Part A trainable params:", count_params(model_a))
rows_a = run_epochs(model_a, train_loader_a, opt_a, 12, test_loader, "A")
acc_a = accuracy(model_a, test_loader)
print("Part A test accuracy:", round(acc_a, 4))
torch.save(model_a.state_dict(), CKPT / "part_a.pt")


## Part B — Rotation self-supervised learning

I use every unlabeled image. Each image is rotated by 0, 90, 180, or 270 degrees, and the network has to predict which one. I train that for 15 epochs.

Then I drop the 4-way rotation head, freeze the encoder, and train only a new linear layer on the same 500 labeled images for 15 epochs.


In [ ]:
class RotationSet(Dataset):
    def __init__(self, base):
        self.base = base
        self.tf = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize(STL_MEAN, STL_STD),
        ])

    def __len__(self):
        return len(self.base)

    def __getitem__(self, i):
        img, _ = self.base[i]
        k = random.randint(0, 3)
        img = img.rotate(k * 90)
        x = self.tf(img)
        return x, k

class RotationNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = make_encoder()
        self.head = nn.Linear(512, 4)

    def forward(self, x):
        return self.head(self.encoder(x))

rot_loader = DataLoader(RotationSet(unlabeled_raw), batch_size=128, shuffle=True, num_workers=2)
torch.manual_seed(SEED)
model_b_pre = RotationNet().to(DEVICE)
opt_b = torch.optim.Adam(model_b_pre.parameters(), lr=1e-3)
print("Part B pretrain params:", count_params(model_b_pre))
rows_b_pre = run_epochs(model_b_pre, rot_loader, opt_b, 15, name="B-rot")
torch.save(model_b_pre.state_dict(), CKPT / "part_b_rotation.pt")

class LinearProbe(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Linear(512, 10)
        for p in self.encoder.parameters():
            p.requires_grad = False

    def train(self, mode=True):
        super().train(mode)
        self.encoder.eval()
        return self

    def forward(self, x):
        with torch.no_grad():
            z = self.encoder(x)
        return self.head(z)

    def embed(self, x):
        self.encoder.eval()
        with torch.no_grad():
            return self.encoder(x)

model_b = LinearProbe(model_b_pre.encoder).to(DEVICE)
opt_b_lin = torch.optim.Adam(model_b.head.parameters(), lr=1e-3)
print("Part B linear params (encoder frozen):", count_params(model_b))
rows_b = run_epochs(model_b, train_loader_a, opt_b_lin, 15, test_loader, "B-lin")
acc_b = accuracy(model_b, test_loader)
print("Part B test accuracy:", round(acc_b, 4))
torch.save({"encoder": model_b_pre.encoder.state_dict(), "head": model_b.head.state_dict()}, CKPT / "part_b_linear.pt")


## Part C — SimCLR-style contrastive learning

I use 20,000 unlabeled images. Each image makes two views. The four augmentations are random resized crop, horizontal flip, color jitter, and Gaussian blur. That is the usual demo set. If the class demo used a different fourth one, swap it here and rerun.

The projection head is a small MLP (512 → 128 → 128). I use cosine similarity and temperature 0.2, and I train for 15 epochs.

Then I drop the projection head, freeze the encoder, and train a linear classifier on the same 500 labeled images for 15 epochs.


In [ ]:
simclr_aug = transforms.Compose([
    transforms.RandomResizedCrop(96, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.4, 0.4, 0.4, 0.1),
    transforms.GaussianBlur(kernel_size=9, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(STL_MEAN, STL_STD),
])

class TwoViewSet(Dataset):
    def __init__(self, base, indices):
        self.base = base
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        img, _ = self.base[self.indices[i]]
        return simclr_aug(img), simclr_aug(img)

class SimCLR(nn.Module):
    def __init__(self, tau=0.2):
        super().__init__()
        self.encoder = make_encoder()
        self.proj = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
        )
        self.tau = tau

    def forward(self, v1, v2):
        z1 = F.normalize(self.proj(self.encoder(v1)), dim=1)
        z2 = F.normalize(self.proj(self.encoder(v2)), dim=1)
        z = torch.cat([z1, z2], dim=0)
        sim = z @ z.T / self.tau
        n = z1.size(0)
        labels = torch.cat([
            torch.arange(n, device=z.device) + n,
            torch.arange(n, device=z.device),
        ])
        sim = sim.masked_fill(torch.eye(2 * n, device=z.device, dtype=torch.bool), -1e9)
        return F.cross_entropy(sim, labels)

    def embed(self, x):
        return self.encoder(x)

sim_loader = DataLoader(TwoViewSet(unlabeled_raw, unlab_perm), batch_size=128, shuffle=True, num_workers=2, drop_last=True)
torch.manual_seed(SEED)
model_c_pre = SimCLR(tau=0.2).to(DEVICE)
opt_c = torch.optim.Adam(model_c_pre.parameters(), lr=1e-3)
print("Part C pretrain params:", count_params(model_c_pre))
rows_c_pre = run_epochs(model_c_pre, sim_loader, opt_c, 15, name="C-simclr", contrastive=True)
torch.save(model_c_pre.state_dict(), CKPT / "part_c_simclr.pt")

model_c = LinearProbe(model_c_pre.encoder).to(DEVICE)
opt_c_lin = torch.optim.Adam(model_c.head.parameters(), lr=1e-3)
print("Part C linear params (encoder frozen):", count_params(model_c))
rows_c = run_epochs(model_c, train_loader_a, opt_c_lin, 15, test_loader, "C-lin")
acc_c = accuracy(model_c, test_loader)
print("Part C test accuracy:", round(acc_c, 4))
torch.save({"encoder": model_c_pre.encoder.state_dict(), "head": model_c.head.state_dict()}, CKPT / "part_c_linear.pt")

print("\n===== test accuracy =====")
print("A supervised:", round(acc_a, 4))
print("B rotation:  ", round(acc_b, 4))
print("C simclr:    ", round(acc_c, 4))


## Part D — Nearest neighbors

I take 3 query images from the test set. The same 3 queries go through all three encoders. Neighbors are the top 5 other test images by cosine similarity. The encoder is frozen. I use the 512-d features, not the projection head and not the class head.


In [ ]:
@torch.no_grad()
def all_embeddings(embed_fn):
    feats = []
    labels = []
    for x, y in test_loader:
        z = embed_fn(x.to(DEVICE))
        z = F.normalize(z, dim=1)
        feats.append(z.cpu())
        labels.append(y)
    return torch.cat(feats), torch.cat(labels)

def show_neighbors(query_ids, banks, path):
    raw = STL10(ROOT, split="test", download=False)
    names = raw.classes
    n_q = len(query_ids)
    fig, axes = plt.subplots(n_q * 3, 6, figsize=(12, 2.2 * n_q * 3))
    row = 0
    for q in query_ids:
        q_img, q_y = raw[q]
        for name, (feats, labels) in banks.items():
            sims = feats @ feats[q]
            sims[q] = -1
            top = torch.topk(sims, 5).indices.tolist()
            axes[row, 0].imshow(q_img)
            axes[row, 0].set_title(f"{name}\nquery {names[q_y]}", fontsize=8)
            axes[row, 0].axis("off")
            for j, idx in enumerate(top):
                img, y = raw[idx]
                axes[row, j + 1].imshow(img)
                axes[row, j + 1].set_title(names[y], fontsize=8)
                axes[row, j + 1].axis("off")
            row += 1
    fig.tight_layout()
    fig.savefig(path, dpi=140)
    plt.show()
    print("saved", path)

g2 = torch.Generator().manual_seed(SEED)
query_ids = torch.randperm(len(test_set), generator=g2)[:3].tolist()
print("query indices:", query_ids)

model_a.eval()
model_b.eval()
model_c.eval()
banks = {
    "Supervised": all_embeddings(model_a.embed),
    "Rotation": all_embeddings(model_b.embed),
    "SimCLR": all_embeddings(model_c.embed),
}
show_neighbors(query_ids, banks, FIG / "nearest_neighbors.png")


## Comparison

I will fill this after the Colab run, using the printed test accuracies. The question is which encoder is best on the 8,000-image test set, why the other two landed where they did, what I would change, and what the neighbor grids say about the features.

What I already know before the numbers:

- Part A sees the real class labels, but only 500 images, so it can memorize that small set and still miss the test set.
- Part B sees all 100,000 unlabeled images, but the job is only "which way is up". That can learn edges and object pose, and it can miss color and class.
- Part C sees 20,000 images twice per step, and the loss pulls two crops of the same photo together. That usually makes better neighbors than rotation, but 20,000 is not the full unlabeled set, and 15 epochs is short for SimCLR.
- The neighbor plot should show whether same-class photos actually sit next to the query. If the neighbors are the right class, the features are useful even when the linear probe is weak.
